In [1]:
#!pip install pandas
#!pip install scikit-learn
#!pip install datasets
#!pip install transformers
#!pip install torch
#!pip install matplotlib
#!pip install seaborn
#!pip install accelerate>=0.26.0

In [ ]:
import os
import ast
from ast import literal_eval
import json

import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments

import pickle

import torch

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
model = "bert-base-multilingual-cased"
model_name = "mBERT"
data = "baseline"
lr_categories = 2e-5
lr_levels = 2e-5
lr_time = 2e-5
num_epochs = 3
batch_size = 16

In [4]:
def combine_df(input_dir):
    dfs = []

    for file in os.listdir(input_dir):
        if file.endswith(".csv"):
            df = pd.read_csv(os.path.join(input_dir, file))
            dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)

    return combined_df

In [5]:
def load_df(input_path):
    file = os.path.basename(input_path)
    if file.endswith(".csv"):
        df = pd.read_csv(input_path)
    return df

In [6]:
category_names = [
    "B1300 Energy level",
    "B140 Attention functions",
    "B152 Emotional functions",
    "B440 Respiration functions",
    "B455 Exercise tolerance functions",
    "B530 Weight maintenance functions",
    "D450 Walking",
    "D550 Eating",
    "D840-D859 Work and employment",
    "B280 Sensations of pain",
    "B134 Sleep functions",
    "D760 Family relationships",
    "B164 Higher-level cognitive functions",
    "D465 Moving around using equipment",
    "D410 Changing basic body position",
    "B230 Hearing functions",
    "D240 Handling stress and other psychological demands",
    "None"
]

In [ ]:
# Create special tokens
special_tokens = [f"[{name}]" for name in category_names] + ["[LEVELS]", "[TEXT]", "[HISTORY]"]

## Categories

### Load the data

In [8]:
categories_input_path = "data_categories/train/train_final_aug.csv"

categories_df = load_df(categories_input_path)

In [ ]:
# Split EHR data in a training and test set
categories_train_df, categories_dev_df = train_test_split(
    categories_df,
    test_size=0.2,
    random_state=523,
    shuffle=True
)

In [10]:
important_columns_categories = ["text", "categories_18", "labels_18"]

categories_train_df = categories_train_df[important_columns_categories]
categories_dev_df = categories_dev_df[important_columns_categories]

In [11]:
# Drop rows with NaN for text
categories_train_df = categories_train_df.dropna(subset=["text"])
categories_dev_df = categories_dev_df.dropna(subset=["text"])

In [12]:
categories_train_df["labels_18"] = categories_train_df["labels_18"].apply(ast.literal_eval)
categories_train_df["labels_18"] = categories_train_df["labels_18"].apply(lambda x: [float(v) for v in x])

categories_dev_df["labels_18"] = categories_dev_df["labels_18"].apply(ast.literal_eval)
categories_dev_df["labels_18"] = categories_dev_df["labels_18"].apply(lambda x: [float(v) for v in x])

### Encode categories

In [13]:
categories_train_df["labels"] = categories_train_df["labels_18"]
categories_dev_df["labels"] = categories_dev_df["labels_18"]

In [14]:
categories_train_dataset = Dataset.from_pandas(categories_train_df)
categories_dev_dataset = Dataset.from_pandas(categories_dev_df)

### Tokenise text with categories

In [15]:
# Load tokeniser
#categories_tokeniser = BertTokenizer.from_pretrained(model)
categories_tokeniser = BertTokenizer.from_pretrained(f"./{data}/{model_name}_categories")

In [16]:
#categories_tokeniser.save_pretrained(f"./{data}/{model_name}_categories")

('./baseline/mBERT_categories/tokenizer_config.json',
 './baseline/mBERT_categories/special_tokens_map.json',
 './baseline/mBERT_categories/vocab.txt',
 './baseline/mBERT_categories/added_tokens.json')

In [ ]:
def tokenise_categories_function(examples):
    return categories_tokeniser(examples["text"], 
                     truncation=True, 
                     padding="max_length", 
                     max_length=512)

In [18]:
"""categories_train_dataset = categories_train_dataset.map(tokenise_categories_function, batched=True)
categories_dev_dataset = categories_dev_dataset.map(tokenise_categories_function, batched=True)"""

Map:   0%|          | 0/272473 [00:00<?, ? examples/s]

Map:   0%|          | 0/68118 [00:00<?, ? examples/s]

In [19]:
"""categories_train_dataset.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)

categories_dev_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)"""

### Train the categories model

In [20]:
def categories_compute_metrics(eval_pred):
    logits, labels = eval_pred

    probabilities = torch.sigmoid(torch.tensor(logits))
    predictions = (probabilities > 0.5).int().numpy()

    return {"micro_f1": f1_score(labels, predictions, average="micro"),
           "macro_f1": f1_score(labels, predictions, average="macro"),
           "weighted_f1": f1_score(labels, predictions, average="weighted")}

In [21]:
"""categories_model = BertForSequenceClassification.from_pretrained(model, 
                                                      num_labels=18,
                                                      problem_type="multi_label_classification")"""
categories_model = BertForSequenceClassification.from_pretrained(f"./{data}/{model_name}_categories")
categories_trainer = Trainer(model=categories_model)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
"""categories_training_args = TrainingArguments(
    output_dir=f"./{data}/results_categories",
    learning_rate=lr_categories,
    num_train_epochs=num_epochs,                    
    per_device_train_batch_size=batch_size,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir=f"./{data}/logs_categories",
)
 
categories_trainer = Trainer(
    model=categories_model,
    args=categories_training_args, 
    train_dataset=categories_train_dataset,
    eval_dataset=categories_dev_dataset,               
    compute_metrics=categories_compute_metrics
)

categories_trainer.train()"""

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Weighted F1
1,0.043700,0.041034,0.863652,0.706187,0.859197
2,0.037100,0.036600,0.877604,0.734523,0.872188
3,0.028400,0.036177,0.881907,0.753434,0.880142


TrainOutput(global_step=51090, training_loss=0.040349233381480804, metrics={'train_runtime': 8011.2108, 'train_samples_per_second': 102.034, 'train_steps_per_second': 6.377, 'total_flos': 2.151028724731269e+17, 'train_loss': 0.040349233381480804, 'epoch': 3.0})

### Saving categories model

In [23]:
#categories_model.save_pretrained(f"./{data}/{model_name}_categories")

## Levels

### Load the data

In [47]:
levels_input_dir_train = r"data_levels/train"
levels_train_df = combine_df(levels_input_dir_train)

In [48]:
levels_input_dir_dev = "data_levels/dev/"
levels_dev_df = combine_df(levels_input_dir_dev)

In [49]:
important_columns_levels = ["text", "categories", "labels"]

levels_train_df = levels_train_df[important_columns_levels]
levels_dev_df = levels_dev_df[important_columns_levels]

In [50]:
# Drop rows with NaN for text
levels_train_df = levels_train_df.dropna(subset=["text"])
levels_dev_df = levels_dev_df.dropna(subset=["text"])

### Augment data 

In [51]:
levels_train_df["labels"] = levels_train_df["labels"].astype(str)
levels_dev_df["labels"] = levels_dev_df["labels"].astype(str)

In [52]:
levels_train_df["categories"] = levels_train_df["categories"].fillna("None")
levels_dev_df["categories"] = levels_dev_df["categories"].fillna("None")

In [63]:
# Combine the predicted categories with the original text for the level model input
levels_train_df["combined_text"] =  f"[{levels_train_df['categories']}] {levels_train_df['text']}"
levels_dev_df["combined_text"] =  f"[{levels_dev_df['categories']}] {levels_dev_df['text']}"

### Encode levels

In [64]:
levels_train_df["labels"] = levels_train_df["labels"].astype(float)
levels_dev_df["labels"] = levels_dev_df["labels"].astype(float)

In [65]:
levels_train_dataset = Dataset.from_pandas(levels_train_df)
levels_dev_dataset = Dataset.from_pandas(levels_dev_df)

### Tokenise text with levels

In [66]:
# Load mBERT tokeniser
#levels_tokeniser = BertTokenizer.from_pretrained(model)
levels_tokeniser = BertTokenizer.from_pretrained(f"./{data}/{model_name}_levels")

In [67]:
levels_tokeniser.add_special_tokens(
    {"additional_special_tokens": special_tokens}
)

21

In [68]:
#levels_tokeniser.save_pretrained(f"./{data}/{model_name}_levels")

('./baseline/mBERT_levels/tokenizer_config.json',
 './baseline/mBERT_levels/special_tokens_map.json',
 './baseline/mBERT_levels/vocab.txt',
 './baseline/mBERT_levels/added_tokens.json')

In [ ]:
def tokenise_levels_function(examples):
    return levels_tokeniser(examples["combined_text"], 
                     truncation=True,  
                     padding="max_length", 
                     max_length=512)

In [70]:
"""levels_train_dataset = levels_train_dataset.map(tokenise_levels_function, batched=True)
levels_dev_dataset = levels_dev_dataset.map(tokenise_levels_function, batched=True)"""

Map:   0%|          | 0/25037 [00:00<?, ? examples/s]

Map:   0%|          | 0/2620 [00:00<?, ? examples/s]

In [71]:
"""levels_train_dataset.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)

levels_dev_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)"""

### Train the levels model

In [72]:
def levels_compute_metrics(eval_pred):
    preds, labels = eval_pred
    predictions = preds.squeeze()

    mae = mean_absolute_error(labels, predictions)
    mse = mean_squared_error(labels, predictions)
    rmse = root_mean_squared_error(labels, predictions)

    return {"mae": mae,
            "mse": mse,
            "rmse": rmse
            }

In [73]:
"""levels_model = BertForSequenceClassification.from_pretrained(model, 
                                                              num_labels=1, #len(level_encoder.classes_),
                                                              problem_type="regression")"""
levels_model = BertForSequenceClassification.from_pretrained(f"./{data}/{model_name}_levels")
levels_model.resize_token_embeddings(len(levels_tokeniser))
levels_trainer = Trainer(model=levels_model)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(119568, 768, padding_idx=0)

In [74]:
"""levels_training_args = TrainingArguments(
    output_dir=f"./{data}/results_levels",
    learning_rate=lr_levels,
    num_train_epochs=num_epochs,                  
    per_device_train_batch_size=batch_size,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir=f"./{data}/logs_levels",
)

levels_trainer = Trainer(
    model=levels_model,
    args=levels_training_args,
    train_dataset=levels_train_dataset,
    eval_dataset=levels_dev_dataset,                
    compute_metrics=levels_compute_metrics
)

levels_trainer.train()"""

Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse
1,1.483500,1.435497,0.981808,1.435497,1.198122
2,1.476900,1.422999,0.990141,1.422999,1.192895
3,1.470600,1.418358,1.005502,1.418358,1.190948


TrainOutput(global_step=4695, training_loss=1.485385024585663, metrics={'train_runtime': 727.5814, 'train_samples_per_second': 103.234, 'train_steps_per_second': 6.453, 'total_flos': 1.976235703932211e+16, 'train_loss': 1.485385024585663, 'epoch': 3.0})

### Saving levels model

In [75]:
#levels_model.save_pretrained(f"./{data}/{model_name}_levels")